# Jianrong — TreeSHAP age-proxy audit

Uses every validation row (month 5) and the same feature matrix for all models.
It never loads test rows. The six-feature proxy set is read from the frozen
`models/proxy_set.json` without modification. Run in Colab with the shared
`CS3268_Project` shortcut in My Drive.

XGBoost's native exact TreeSHAP explains the raw margin (log odds). For
ExpGrad, component SHAP vectors are combined with the saved mixture weights
*before* taking absolute values. This weighted raw-margin explanation is an
approximation to the deployed weighted-probability mixture.

In [ ]:
from pathlib import Path
import json
import os
import hashlib
import shutil
import numpy as np
import pandas as pd
import xgboost as xgb

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # A local run may set ROOT to a local copy of CS3268_Project.

ROOT = Path(os.environ.get('CS3268_ROOT',
                           '/content/drive/MyDrive/CS3268_Project'))
DATA = ROOT / 'data'
MODELS = ROOT / 'models'
PREDS = ROOT / 'preds'
OUT = MODELS / 'jianrong_treeshap_audit'
CHUNK = 5000
PROXY_FILE = (MODELS / 'proxy_set.json' if (MODELS / 'proxy_set.json').exists()
              else OUT / 'proxy_set_snapshot.json')

required = [DATA / 'base.parquet', PROXY_FILE,
            MODELS / 'baseline.json', MODELS / 'reweigh.json',
            MODELS / 'expgrad_weights.csv',
            MODELS / 'expgrad_components',
            MODELS / 'expgrad_db0.01' / 'weights.csv']
missing = [str(p) for p in required if not p.exists()]
assert not missing, 'Missing project files:\n' + '\n'.join(missing)

# Parquet filtering returns only month 5; do not call load_split(), which reads
# the whole base file before filtering.
val = pd.read_parquet(DATA / 'base.parquet', filters=[('month', '==', 5)])
assert len(val) == 119323 and val['month'].eq(5).all()
assert val['id'].is_unique
X = val.drop(columns=['id', 'fraud_bool', 'month', 'age_group'])
features = X.columns.tolist()
assert len(features) == 30 and 'customer_age' in features
assert val['age_group'].eq(val['customer_age'].ge(50).astype(int)).all()

with open(PROXY_FILE, encoding='utf-8') as f:
    frozen = json.load(f)
proxy_sha256 = hashlib.sha256(PROXY_FILE.read_bytes()).hexdigest()
assert frozen['threshold'] == 0.60
proxy_set = frozen['proxy_set']
assert len(proxy_set) == 6 and set(proxy_set) <= set(features)
print(f'Validation rows: {len(X):,}; features: {len(features)}; proxy set: {proxy_set}')

## Load saved models and verify their validation scores

Validation uses the prediction files as a model-artifact check. It reads only
month 5 rows and does not use saved decisions for attribution.

In [ ]:
def load_booster(path):
    booster = xgb.Booster(model_file=str(path))
    assert booster.feature_names == features, f'Feature mismatch in {path}'
    return booster


def batches():
    for start in range(0, len(X), CHUNK):
        yield start, xgb.DMatrix(X.iloc[start:start + CHUNK],
                                 enable_categorical=True)


def probabilities(booster):
    return np.concatenate([booster.predict(dm, validate_features=True)
                           for _, dm in batches()])


def check_saved_scores(name, score, file_name, atol=1e-5):
    saved = pd.read_parquet(PREDS / file_name,
                            filters=[('month', '==', 5)],
                            columns=['id', 'month', 'score'])
    assert saved['month'].eq(5).all() and saved['id'].is_unique
    aligned = val[['id']].merge(saved[['id', 'score']], on='id',
                                how='left', validate='one_to_one', sort=False)
    assert aligned['score'].notna().all() and len(aligned) == len(val)
    err = np.max(np.abs(score - aligned['score'].to_numpy()))
    assert np.allclose(score, aligned['score'].to_numpy(),
                       atol=atol, rtol=0), f'{name} scores differ: max {err}'
    print(f'{name}: saved score check OK (max absolute difference {err:.2g})')


def mixture(directory, weights_file):
    w = pd.read_csv(weights_file)
    assert {'component', 'weight'} <= set(w.columns)
    assert (w['weight'] > 0).all() and w['component'].is_unique
    boosters = [load_booster(directory / f'comp_{int(i)}.json')
                for i in w['component']]
    weights = w['weight'].to_numpy(dtype=float, copy=True)
    weights /= weights.sum()
    return boosters, weights


baseline = load_booster(MODELS / 'baseline.json')
reweigh = load_booster(MODELS / 'reweigh.json')
eg05_models, eg05_weights = mixture(MODELS / 'expgrad_components',
                                    MODELS / 'expgrad_weights.csv')
eg01_models, eg01_weights = mixture(MODELS / 'expgrad_db0.01',
                                    MODELS / 'expgrad_db0.01' / 'weights.csv')

check_saved_scores('baseline', probabilities(baseline),
                   'preds_baseline.parquet')
check_saved_scores('reweigh', probabilities(reweigh),
                   'preds_reweigh.parquet')
check_saved_scores('ExpGrad 0.05',
                   sum(w * probabilities(b) for b, w in zip(eg05_models, eg05_weights)),
                   'preds_expgrad.parquet')
check_saved_scores('ExpGrad 0.01',
                   sum(w * probabilities(b) for b, w in zip(eg01_models, eg01_weights)),
                   'preds_expgrad_db0.01.parquet')

# Group thresholds use exactly the baseline score. Verify that fact on val.
group_scores = pd.read_parquet(PREDS / 'preds_groupthr.parquet',
                               filters=[('month', '==', 5)],
                               columns=['id', 'month', 'score'])
assert group_scores['month'].eq(5).all() and group_scores['id'].is_unique
group_aligned = val[['id']].merge(group_scores[['id', 'score']], on='id',
                                 how='left', validate='one_to_one', sort=False)
assert np.allclose(probabilities(baseline), group_aligned['score'].to_numpy(),
                   atol=1e-5, rtol=0)
print('Group-threshold scores equal baseline scores; no separate SHAP run.')

## Exact native TreeSHAP on each saved tree model

`pred_contribs=True` returns one contribution per feature plus a bias column
in raw-margin units. We combine signed component contributions row by row.

In [ ]:
def mean_abs_contributions(boosters, weights):
    sums = np.zeros(len(features), dtype=np.float64)
    n = 0
    for _, dm in batches():
        combined = None
        for booster, weight in zip(boosters, weights):
            contrib = booster.predict(dm, pred_contribs=True,
                                      approx_contribs=False,
                                      validate_features=True)
            assert contrib.shape == (dm.num_row(), len(features) + 1)
            current = weight * contrib[:, :-1].astype(np.float64)
            combined = current if combined is None else combined + current
        sums += np.abs(combined).sum(axis=0)
        n += dm.num_row()
    assert n == len(X)
    return pd.Series(sums / n, index=features)


specs = {
    'baseline': ([baseline], [1.0]),
    'reweigh': ([reweigh], [1.0]),
    'expgrad_0.05': (eg05_models, eg05_weights),
    'expgrad_0.01': (eg01_models, eg01_weights),
}
rows = []
for model_name, (boosters, weights) in specs.items():
    mean_abs = mean_abs_contributions(boosters, weights)
    share = mean_abs / mean_abs.sum()
    rows.extend({'model': model_name, 'feature': feature,
                 'mean_abs_shap_raw_margin': mean_abs[feature],
                 'share': share[feature],
                 'in_frozen_proxy_set': feature in proxy_set}
                for feature in features)
    print(model_name, 'proxy share:', share.loc[proxy_set].sum(),
          'customer_age share:', share.loc['customer_age'])

detail = pd.DataFrame(rows)
baseline_share = detail[detail.model == 'baseline'].set_index('feature')['share']
detail['change_vs_baseline_pp'] = detail.apply(
    lambda r: 100 * (r['share'] - baseline_share[r['feature']]), axis=1)

summary = (detail[detail.in_frozen_proxy_set]
           .groupby('model')['share'].sum().rename('proxy_share').to_frame())
summary['customer_age_share'] = (detail[detail.feature == 'customer_age']
                                 .set_index('model')['share'])
summary['proxy_change_vs_baseline_pp'] = 100 * (
    summary['proxy_share'] - summary.loc['baseline', 'proxy_share'])
summary['customer_age_change_vs_baseline_pp'] = 100 * (
    summary['customer_age_share'] - summary.loc['baseline', 'customer_age_share'])
summary.loc['groupthr'] = summary.loc['baseline']
summary = summary.reset_index()

deltas = pd.concat([
    pd.concat([g[g.change_vs_baseline_pp > 0.01]
               .nlargest(5, 'change_vs_baseline_pp').assign(direction='gain'),
               g[g.change_vs_baseline_pp < -0.01]
               .nsmallest(5, 'change_vs_baseline_pp').assign(direction='loss')])
    for model, g in detail[detail.model != 'baseline'].groupby('model')
], ignore_index=True)

# Post-freeze sensitivity uses AUCs already saved in the frozen file; it does
# not alter the 0.60 definition or the primary comparison.
sensitivity = []
for threshold in [0.55, 0.60, 0.65, 0.70]:
    selected = [f for f, auc in frozen['all_feature_auc'].items()
                if auc >= threshold]
    for model, group in detail.groupby('model'):
        sensitivity.append({'threshold_post_freeze': threshold,
                            'model': model, 'n_features': len(selected),
                            'proxy_share': group[group.feature.isin(selected)]['share'].sum()})
sensitivity = pd.DataFrame(sensitivity)

OUT.mkdir(parents=True, exist_ok=True)
snapshot = OUT / 'proxy_set_snapshot.json'
if PROXY_FILE.resolve() != snapshot.resolve():
    shutil.copyfile(PROXY_FILE, snapshot)
detail.to_csv(OUT / 'feature_shares.csv', index=False)
summary.to_csv(OUT / 'model_summary.csv', index=False)
deltas.to_csv(OUT / 'top_gains_losses.csv', index=False)
sensitivity.to_csv(OUT / 'post_freeze_sensitivity.csv', index=False)
focus = detail[detail.feature.isin(proxy_set + ['customer_age'])].copy()
focus.to_csv(OUT / 'proxy_and_age_rows.csv', index=False)

def markdown_table(columns, records):
    return ('| ' + ' | '.join(columns) + ' |\n'
            + '| ' + ' | '.join(['---'] * len(columns)) + ' |\n'
            + ''.join('| ' + ' | '.join(map(str, row)) + ' |\n'
                      for row in records))

summary_rows = [(r.model, f'{100*r.proxy_share:.2f}%',
                 f'{r.proxy_change_vs_baseline_pp:+.2f}',
                 f'{100*r.customer_age_share:.2f}%')
                for r in summary.itertuples()]
si = summary.set_index('model')
finding = (
    f"Against baseline, reweighing changes frozen-proxy share by "
    f"{si.loc['reweigh', 'proxy_change_vs_baseline_pp']:+.2f} percentage points; "
    f"ExpGrad 0.05 changes it by "
    f"{si.loc['expgrad_0.05', 'proxy_change_vs_baseline_pp']:+.2f}; "
    f"ExpGrad 0.01 changes it by "
    f"{si.loc['expgrad_0.01', 'proxy_change_vs_baseline_pp']:+.2f}. "
    f"Direct `customer_age` share changes from "
    f"{100*si.loc['baseline', 'customer_age_share']:.2f}% at baseline to "
    f"{100*si.loc['expgrad_0.05', 'customer_age_share']:.2f}% for ExpGrad 0.05 "
    f"and {100*si.loc['expgrad_0.01', 'customer_age_share']:.2f}% for "
    f"ExpGrad 0.01. The small proxy-share increase for the 0.05 winner is "
    f"descriptive evidence, not proof of bias relocation."
)
feature_rows = []
for feature in proxy_set + ['customer_age']:
    feature_rows.append((feature, *(
        f'{100*detail[(detail.model == model) & (detail.feature == feature)].share.iloc[0]:.2f}%'
        for model in ['baseline', 'reweigh', 'expgrad_0.05', 'expgrad_0.01'])))
delta_rows = [(r.model, r.direction, r.feature,
               f'{r.change_vs_baseline_pp:+.2f}') for r in deltas.itertuples()]
sens_rows = [(f'{r.threshold_post_freeze:.2f}', r.model, r.n_features,
              f'{100*r.proxy_share:.2f}%') for r in sensitivity.itertuples()]

report = '''# Jianrong TreeSHAP audit — validation month 5

Every model was explained on the same 119,323 validation rows using its saved
XGBoost trees and the frozen six-feature proxy set. Test months were excluded.
Saved validation predictions were checked against each loaded model.
Frozen proxy JSON SHA-256: `''' + proxy_sha256 + '''`.

## Shares of total mean absolute SHAP

''' + markdown_table(
    ['Model', 'Frozen proxy share', 'Change vs baseline (pp)', 'customer_age share'],
    summary_rows) + '\n' + finding + '''
Group thresholds use the baseline scores, so their attribution is the baseline's
by construction. No separate group-threshold SHAP run was done.

## Frozen proxies and customer_age, each as its own row

''' + markdown_table(
    ['Feature', 'Baseline', 'Reweigh', 'ExpGrad 0.05', 'ExpGrad 0.01'],
    feature_rows) + '''
## Largest attribution-share changes versus baseline

These are descriptive changes in normalized share, in percentage points.

''' + markdown_table(
    ['Model', 'Direction', 'Feature', 'Change (pp)'], delta_rows) + '''
## Post-freeze threshold sensitivity

The primary proxy set remains the frozen 0.60 set. These alternative cutoffs
were evaluated after the freeze and do not redefine the primary analysis.

''' + markdown_table(
    ['AUC cutoff', 'Model', 'Feature count', 'Share'], sens_rows) + '''
## Interpretation and limits

XGBoost native exact TreeSHAP explains raw margins (log odds). The shares divide
each feature's mean absolute contribution by the total across all 30 features.
For ExpGrad, saved component weights combine signed SHAP values for each row
before taking absolute values. That is an approximation for the deployed score,
which averages component probabilities rather than raw margins. Compare shares
across models; raw SHAP magnitudes are not the comparison target.

Correlated features can split SHAP credit and understate proxy use. The frozen
screen tests one feature at a time, so it can miss joint proxies. Attribution
shares are descriptive and do not establish causal discrimination or prove
that age information moved from one feature to another. The 0.60 cutoff was
fixed before inspecting these SHAP results and was not changed afterward.
'''
(OUT / 'report.md').write_text(report, encoding='utf-8')
print(summary.to_string(index=False, float_format=lambda z: f'{z:.4f}'))
print('Saved to', OUT)

## Report text and interpretation

Use the printed numbers in the report. A larger proxy share is descriptive
evidence of reliance on the frozen age-linked features. It does not establish
causal discrimination or show that age information moved from one feature to
another. Correlated features can divide SHAP credit, understating proxy use;
single-feature proxy screening can miss joint proxies. The 0.60 threshold was
frozen before this analysis and must not be moved after seeing its result.
The 0.55/0.65/0.70 table is explicitly post-freeze sensitivity.

Group thresholds change only decisions, not scores: their attribution is
baseline's by construction. ExpGrad combines component raw-margin SHAP values
using saved weights, which approximates attribution of the deployed weighted
probability mixture.